In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

file_path = r"\\cabinet\derivatives\DeepDR\FVs_medsiglip"

data_dict = torch.load(file_path, map_location='cpu', weights_only=False)



In [2]:
from collections import defaultdict, Counter

# data_dict: 例如 {"1_l1": ..., "1_l2": ..., "1_r1": ..., ...}

eye_count = defaultdict(lambda: {"l": 0, "r": 0})
view_detail = defaultdict(list)

for k in data_dict.keys():
    # 例如 k = "12_l1"
    parts = k.split("_")
    if len(parts) != 2:
        print(f"格式异常: {k}")
        continue

    pid, eyeview = parts
    eye = eyeview[0]   # l 或 r
    view = eyeview[1:] # 1 或 2

    if eye in ["l", "r"]:
        eye_count[pid][eye] += 1
        view_detail[pid].append(eyeview)
    else:
        print(f"未知眼别: {k}")

# 1. 全局左右眼总数
total_left = sum(v["l"] for v in eye_count.values())
total_right = sum(v["r"] for v in eye_count.values())

print("全局统计：")
print(f"左眼总数 = {total_left}")
print(f"右眼总数 = {total_right}")
print(f"左右眼总数是否相等: {total_left == total_right}")

# 2. 每个编号是否左右一对一
not_one_to_one = []
for pid, cnt in eye_count.items():
    if cnt["l"] != cnt["r"]:
        not_one_to_one.append((pid, cnt["l"], cnt["r"], sorted(view_detail[pid])))

print("\n每个编号左右是否一对一：")
print(f"总编号数 = {len(eye_count)}")
print(f"左右数量不相等的编号数 = {len(not_one_to_one)}")

# 显示前20个异常编号
for item in not_one_to_one[:20]:
    pid, lcnt, rcnt, views = item
    print(f"id={pid}, left={lcnt}, right={rcnt}, views={views}")

# 3. 看看常见模式
pattern_counter = Counter()
for pid, views in view_detail.items():
    pattern_counter[tuple(sorted(views))] += 1

print("\n最常见的视角组合模式（前10个）：")
for pattern, cnt in pattern_counter.most_common(10):
    print(pattern, cnt)

全局统计：
左眼总数 = 599
右眼总数 = 601
左右眼总数是否相等: False

每个编号左右是否一对一：
总编号数 = 300
左右数量不相等的编号数 = 1
id=164, left=1, right=3, views=['l3', 'r1', 'r3', 'r4']

最常见的视角组合模式（前10个）：
('l1', 'l2', 'r1', 'r2') 296
('l1', 'l3', 'r2', 'r3') 1
('l3', 'l4', 'r3', 'r4') 1
('l3', 'r1', 'r3', 'r4') 1
('l2', 'l3', 'r2', 'r3') 1


In [3]:
from collections import defaultdict, Counter

# 1. 按编号收集后缀
id_to_suffixes = defaultdict(list)
for k in data_dict.keys():
    pid, suffix = k.split("_", 1)
    id_to_suffixes[pid].append(suffix)

# 2. 定义标准模式
expected = ['l1', 'l2', 'r1', 'r2']

# 3. 找出异常编号
bad_ids = {}
for pid, suffixes in id_to_suffixes.items():
    suffixes_sorted = sorted(suffixes)
    if suffixes_sorted != expected:
        bad_ids[pid] = suffixes_sorted

print("总编号数:", len(id_to_suffixes))
print("异常编号数:", len(bad_ids))
print()

# 4. 逐个打印异常编号
print("=== 异常编号详情 ===")
for pid in sorted(bad_ids, key=lambda x: int(x)):
    raw_keys = sorted([k for k in data_dict.keys() if k.startswith(f"{pid}_")])
    print(f"id={pid}")
    print(f"  suffixes = {bad_ids[pid]}")
    print(f"  raw_keys  = {raw_keys}")
    print()

# 5. 统计异常模式出现次数
pattern_counter = Counter(tuple(v) for v in bad_ids.values())

print("=== 异常模式统计 ===")
for pattern, cnt in pattern_counter.most_common():
    print(f"{pattern} : {cnt}")

总编号数: 300
异常编号数: 4

=== 异常编号详情 ===
id=56
  suffixes = ['l1', 'l3', 'r2', 'r3']
  raw_keys  = ['56_l1', '56_l3', '56_r2', '56_r3']

id=77
  suffixes = ['l3', 'l4', 'r3', 'r4']
  raw_keys  = ['77_l3', '77_l4', '77_r3', '77_r4']

id=164
  suffixes = ['l3', 'r1', 'r3', 'r4']
  raw_keys  = ['164_l3', '164_r1', '164_r3', '164_r4']

id=167
  suffixes = ['l2', 'l3', 'r2', 'r3']
  raw_keys  = ['167_l2', '167_l3', '167_r2', '167_r3']

=== 异常模式统计 ===
('l1', 'l3', 'r2', 'r3') : 1
('l3', 'l4', 'r3', 'r4') : 1
('l3', 'r1', 'r3', 'r4') : 1
('l2', 'l3', 'r2', 'r3') : 1


In [4]:
bad_ids = {"56", "77", "164", "167"}

data_dict_clean = {
    k: v for k, v in data_dict.items()
    if k.split("_", 1)[0] not in bad_ids
}

print("原始 key 数:", len(data_dict))
print("清洗后 key 数:", len(data_dict_clean))
print("删除的编号:", sorted(bad_ids, key=int))

原始 key 数: 1200
清洗后 key 数: 1184
删除的编号: ['56', '77', '164', '167']


In [10]:
from collections import defaultdict

id_to_suffixes = defaultdict(list)
for k in data_dict_clean.keys():
    pid, suffix = k.split("_", 1)
    id_to_suffixes[pid].append(suffix)

expected = ['l1', 'l2', 'r1', 'r2']
bad_after_clean = []

for pid, suffixes in id_to_suffixes.items():
    if sorted(suffixes) != expected:
        bad_after_clean.append((pid, sorted(suffixes)))

print("清洗后异常编号数:", len(bad_after_clean))
print(bad_after_clean[:10])

if len(bad_after_clean)==0:
    torch.save(data_dict_clean, "./DR_dict_clean(56,77,164,167).pt")

清洗后异常编号数: 0
[]


In [9]:
import numpy as np
import torch

# 先试 numpy
try:
    obj = np.load(file_path, allow_pickle=True)
    print("NumPy 可以加载")
    print("类型:", type(obj))
    if hasattr(obj, "files"):
        print("keys/files:", obj.files[:10])
except Exception as e:
    print("NumPy 加载失败:", e)

# 再试 torch
try:
    obj = torch.load(file_path)
    print("PyTorch 可以加载")
    print("类型:", type(obj))
except Exception as e:
    print("PyTorch 加载失败:", e)

NumPy 可以加载
类型: <class 'numpy.lib.npyio.NpzFile'>
keys/files: ['FVs_medsiglip/data.pkl', 'FVs_medsiglip/.format_version', 'FVs_medsiglip/.storage_alignment', 'FVs_medsiglip/byteorder', 'FVs_medsiglip/version', 'FVs_medsiglip/.data/serialization_id']
PyTorch 加载失败: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_s